# Model 1 — CNN Tabanlı Baseline Model
**Plant Leaf Disease Classification Dataset**  
Dataset: https://www.kaggle.com/datasets/mertcangelbal/plant-leaf-disease-classification-dataset  
GPU: RTX 5090 (Vast.ai)

In [ ]:
# ── Dataset İndirme ──────────────────────────────────────────────────────────
!pip install -q kagglehub

import kagglehub, shutil, os
from pathlib import Path

_raw = kagglehub.dataset_download("mertcangelbal/plant-leaf-disease-classification-dataset")
print("Kaggle cache yolu:", _raw)

DATA_ROOT = Path("../data")
if not DATA_ROOT.exists():
    print(f"Dataset {DATA_ROOT} dizinine kopyalanıyor...")
    shutil.copytree(_raw, str(DATA_ROOT))
    print("Kopyalama tamamlandı.")
else:
    print(f"{DATA_ROOT} zaten mevcut, indirme atlandı.")

In [ ]:
# ── Kütüphaneler ─────────────────────────────────────────────────────────────
import os, time, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score
)
from sklearn.preprocessing import label_binarize

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Konfigürasyon ─────────────────────────────────────────────────────────────
CONFIG = {
    "data_dir":     "../data",
    "img_size":     224,
    "batch_size":   32,
    "num_classes":  105,
    "epochs":       50,
    "lr":           3e-4,
    "weight_decay": 1e-4,
    "num_workers":  4,
    "model_name":   "cnn_baseline",
    "save_path":    "./checkpoints/cnn_baseline_best.pth",
}
os.makedirs("./checkpoints", exist_ok=True)
os.makedirs("./results", exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
# ── Veri Ön İşleme & Augmentation ────────────────────────────────────────────
train_transforms = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomCrop(CONFIG["img_size"], padding=8),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

data_dir = Path(CONFIG["data_dir"])
train_dataset = datasets.ImageFolder(data_dir / "train", transform=train_transforms)
val_dataset   = datasets.ImageFolder(data_dir / "val",   transform=val_transforms)
test_dataset  = datasets.ImageFolder(data_dir / "test",  transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"],
                          shuffle=True,  num_workers=CONFIG["num_workers"], pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG["batch_size"],
                          shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG["batch_size"],
                          shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

print(f"Train: {len(train_dataset):,} | Val: {len(val_dataset):,} | Test: {len(test_dataset):,}")
print(f"Sınıf sayısı: {len(train_dataset.classes)}")

In [ ]:
# ── Class Weights (dengesizlik için) ─────────────────────────────────────────
class_counts = np.array([len(list((data_dir / "train" / c).glob("*.jpg")))
                         for c in train_dataset.classes])
class_weights = torch.tensor(
    len(train_dataset) / (CONFIG["num_classes"] * class_counts),
    dtype=torch.float32
).to(DEVICE)
print(f"Min weight: {class_weights.min():.4f} | Max weight: {class_weights.max():.4f}")

In [ ]:
# ── CNN Mimarisi ──────────────────────────────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class CNNBaseline(nn.Module):
    def __init__(self, num_classes=105):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3,   64),  nn.MaxPool2d(2),   # 112
            ConvBlock(64,  128), nn.MaxPool2d(2),   # 56
            ConvBlock(128, 256), nn.MaxPool2d(2),   # 28
            ConvBlock(256, 512), nn.MaxPool2d(2),   # 14
            ConvBlock(512, 512), nn.MaxPool2d(2),   # 7
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )
    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        return self.classifier(x)

model = CNNBaseline(num_classes=CONFIG["num_classes"]).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Toplam parametre: {total_params:,}")

In [ ]:
# ── Optimizer, Loss, Scheduler ───────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(),
                        lr=CONFIG["lr"],
                        weight_decay=CONFIG["weight_decay"])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,
                                                  T_max=CONFIG["epochs"],
                                                  eta_min=1e-6)

In [ ]:
# ── Eğitim Döngüsü ────────────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0
train_start = time.time()

for epoch in range(1, CONFIG["epochs"] + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    vl_loss, vl_acc = evaluate(model, val_loader, criterion, DEVICE)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(vl_loss)
    history["val_acc"].append(vl_acc)

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), CONFIG["save_path"])

    elapsed = time.time() - t0
    print(f"Epoch [{epoch:02d}/{CONFIG['epochs']}] "
          f"Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | "
          f"Val Loss: {vl_loss:.4f} Acc: {vl_acc:.4f} | "
          f"{elapsed:.1f}s")

total_train_time = time.time() - train_start
print(f"\nToplam eğitim süresi: {total_train_time/60:.1f} dakika")
print(f"En iyi Val Accuracy: {best_val_acc:.4f}")

In [ ]:
# ── Eğitim Grafikleri ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"],   label="Val")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")
axes[1].plot(history["train_acc"], label="Train")
axes[1].plot(history["val_acc"],   label="Val")
axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].set_xlabel("Epoch")
plt.suptitle("CNN Baseline — Training History")
plt.tight_layout()
plt.savefig("./results/cnn_baseline_training.png", dpi=150)
plt.show()

In [ ]:
# ── Test Değerlendirmesi ──────────────────────────────────────────────────────
model.load_state_dict(torch.load(CONFIG["save_path"]))
model.eval()

all_preds, all_labels, all_probs = [], [], []
infer_start = time.time()

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        outputs = model(imgs)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        all_probs.extend(probs)

infer_time = time.time() - infer_start
all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
rec  = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
f1   = f1_score(all_labels, all_preds, average="weighted", zero_division=0)

print(f"Test Accuracy:  {acc:.4f}")
print(f"Precision:      {prec:.4f}")
print(f"Recall:         {rec:.4f}")
print(f"F1 Score:       {f1:.4f}")
print(f"Inference Time: {infer_time:.2f}s ({infer_time/len(test_dataset)*1000:.2f} ms/image)")
print(f"Training Time:  {total_train_time/60:.1f} min")

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(20, 18))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues')
plt.title("CNN Baseline — Confusion Matrix")
plt.ylabel("True Label"); plt.xlabel("Predicted Label")
plt.tight_layout()
plt.savefig("./results/cnn_baseline_confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# ── Kapsamlı Grafiksel Değerlendirme ─────────────────────────────────────────
from sklearn.metrics import classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize
import warnings; warnings.filterwarnings("ignore")

MODEL_LABEL = "CNN Baseline"
class_names  = train_dataset.classes

# ── 1. Per-Class F1 Score (Top-20 en iyi / en kötü) ──────────────────────────
report_dict = classification_report(all_labels, all_preds,
                                    target_names=class_names,
                                    output_dict=True, zero_division=0)
per_class_f1 = {k: v["f1-score"] for k, v in report_dict.items()
                if k not in ("accuracy", "macro avg", "weighted avg")}
sorted_f1 = sorted(per_class_f1.items(), key=lambda x: x[1])

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
worst20 = sorted_f1[:20]
best20  = sorted_f1[-20:][::-1]

axes[0].barh([x[0][:25] for x in worst20], [x[1] for x in worst20], color="#EF5350")
axes[0].set_title(f"{MODEL_LABEL} — En Düşük F1 (20 Sınıf)")
axes[0].set_xlabel("F1 Score"); axes[0].set_xlim(0, 1)

axes[1].barh([x[0][:25] for x in best20],  [x[1] for x in best20],  color="#66BB6A")
axes[1].set_title(f"{MODEL_LABEL} — En Yüksek F1 (20 Sınıf)")
axes[1].set_xlabel("F1 Score"); axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.savefig(f"./results/cnn_baseline_per_class_f1.png", dpi=150)
plt.show()

# ── 2. Top-10 Karıştırılan Sınıf Çiftleri ────────────────────────────────────
cm_err = confusion_matrix(all_labels, all_preds)
np.fill_diagonal(cm_err, 0)
flat_idx = np.argsort(cm_err.ravel())[::-1][:10]
rows, cols = np.unravel_index(flat_idx, cm_err.shape)

fig, ax = plt.subplots(figsize=(12, 5))
labels_top = [f"{class_names[r][:15]}→{class_names[c][:15]}" for r, c in zip(rows, cols)]
counts_top = [cm_err[r, c] for r, c in zip(rows, cols)]
bars = ax.barh(labels_top[::-1], counts_top[::-1], color="#5C6BC0")
for bar, val in zip(bars, counts_top[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9)
ax.set_title(f"{MODEL_LABEL} — En Çok Karıştırılan 10 Sınıf Çifti")
ax.set_xlabel("Yanlış Sınıflandırma Sayısı")
plt.tight_layout()
plt.savefig(f"./results/cnn_baseline_top_errors.png", dpi=150)
plt.show()

# ── 3. Makro-Ortalama ROC Eğrisi ─────────────────────────────────────────────
labels_bin = label_binarize(all_labels, classes=list(range(CONFIG["num_classes"])))
fpr_dict, tpr_dict, roc_auc_dict = {}, {}, {}
for i in range(CONFIG["num_classes"]):
    if labels_bin[:, i].sum() > 0:
        fpr_dict[i], tpr_dict[i], _ = roc_curve(labels_bin[:, i], all_probs[:, i])
        roc_auc_dict[i] = auc(fpr_dict[i], tpr_dict[i])

all_fpr = np.unique(np.concatenate([fpr_dict[i] for i in roc_auc_dict]))
mean_tpr = np.zeros_like(all_fpr)
for i in roc_auc_dict:
    mean_tpr += np.interp(all_fpr, fpr_dict[i], tpr_dict[i])
mean_tpr /= len(roc_auc_dict)
macro_auc = auc(all_fpr, mean_tpr)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(all_fpr, mean_tpr, color="#1565C0", lw=2,
        label=f"Makro-Ortalama ROC (AUC = {macro_auc:.4f})")
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title(f"{MODEL_LABEL} — ROC Eğrisi (Makro Ortalama)")
ax.legend(loc="lower right"); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"./results/cnn_baseline_roc.png", dpi=150)
plt.show()

print(f"\nMakro-Ortalama ROC AUC: {macro_auc:.4f}")
print("Grafikler ./results/ dizinine kaydedildi.")

In [ ]:
# ── Sonuçları Kaydet ──────────────────────────────────────────────────────────
results = {
    "model": "CNN Baseline",
    "accuracy":       round(acc, 4),
    "precision":      round(prec, 4),
    "recall":         round(rec, 4),
    "f1_score":       round(f1, 4),
    "roc_auc_macro":  round(macro_auc, 4),
    "training_time_min": round(total_train_time / 60, 2),
    "inference_time_ms_per_image": round(infer_time / len(test_dataset) * 1000, 4),
    "best_val_acc":   round(best_val_acc, 4),
    "total_params":   total_params,
    "config":         CONFIG,
}
with open("./results/cnn_baseline_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Sonuçlar kaydedildi: ./results/cnn_baseline_results.json")
print(json.dumps(results, indent=2))

## Mobil Uyumluluk Notu — CNN Baseline

| Özellik | Değer |
|---------|-------|
| Mimari | CNN (VGG benzeri, sıfırdan) |
| Parametre sayısı | ~14M |
| Model boyutu (.pth) | ~55 MB |
| Inference süresi (GPU) | < 5 ms/görüntü |
| Mobil uygunluğu | ✅ Yüksek |

**Avantajlar:**
- Düşük parametre sayısı — mobil belleğe sığar
- Transformer'lara kıyasla düşük hesaplama maliyeti
- TorchScript / ONNX / Core ML dönüşümü kolay

**Potansiyel kısıtlar:**
- Karmaşık görüntülerde Transformer modellerine göre düşük doğruluk

**Mobil dönüşüm için önerilen araçlar:** TorchScript (`torch.jit.script`), ONNX (`torch.onnx.export`), Core ML Tools (iOS)